# Carta Social: data collection

Collect equipment pages from the Carta Social portal into versioned snapshots,
rebuild service tables from a chosen snapshot, and inspect the result.
Extraction is implemented in scripts/scraper.py and snapshot management in
scripts/snapshots.py.

The default mode makes no HTTP requests. Data paths come from root config.toml.
This notebook loads root .env before using the configured HTTP identity.

Each record describes a social service offered by a facility. A facility may
offer several services, so service and facility counts differ.

In [1]:
from pathlib import Path
import sys

# Support execution from the repository root or the notebooks directory.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "scripts" / "utils.py").exists():
    raise FileNotFoundError("Start Jupyter from the ScraperCartaSocial repository.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [4]:
import json
import os
import pandas as pd

from scripts.utils import INPUTS, ensure_data_directories, load_environment
from scripts.scraper import export_records, run_scraper
from scripts.snapshots import (create_snapshot, export_dir_for, list_snapshots,
                               read_manifest, resolve_snapshot)
from scripts.cleaning import standardize_columns

load_environment()
ensure_data_directories()

TOMLDecodeError: Invalid value (at line 4, column 13)

## Collection scope

Equipment identifiers provide access to portal records; they do not define
a random sample or establish complete coverage. Gaps, failed requests, and
portal changes must be distinguished from the absence of a facility.

Before enabling collection, review the portal's terms and select an
appropriate request rate. The initial range is limited to five IDs.
The HTML selectors originate from the supplied implementation and may
need revision if the portal changes.

## Run options

| Option | Values |
| --- | --- |
| `collection_mode` | `"none"`: no HTTP requests, use stored snapshots · `"new"`: collect into a new snapshot (earlier snapshots are kept) · `"resume"`: continue the unfinished snapshot `snapshot_id`, retrying failed IDs |
| `snapshot_id` | `"latest"` or an identifier such as `"20261118T151700Z"`; used for resume, rebuild, and inspection. A `"new"` collection selects its own snapshot |
| `rebuild_csv` | `True` regenerates the snapshot's CSVs; they are also built automatically when missing |
| `rebuild_from` | `"json"`: parsed records stored at collection time · `"html"`: re-parse stored pages with the current parser |
| `inspect_source` | `"snapshot"`: CSVs of the selected snapshot · `"config"`: `inputs.services` and `inputs.equipment_urls` from config.toml |

In [ ]:
collection_mode = "none"     # "none" | "new" | "resume"
first_id = 1
last_id = 5
request_delay_seconds = 5
snapshot_label = None        # optional suffix for new snapshots, e.g. "pilot"

snapshot_id = "latest"       # "latest" or an exact snapshot identifier
rebuild_csv = False
rebuild_from = "json"        # "json" | "html"

inspect_source = "config"    # "snapshot" | "config"

if collection_mode not in {"none", "new", "resume"}:
    raise ValueError("collection_mode must be 'none', 'new', or 'resume'.")
if rebuild_from not in {"json", "html"}:
    raise ValueError("rebuild_from must be 'json' or 'html'.")
if inspect_source not in {"snapshot", "config"}:
    raise ValueError("inspect_source must be 'snapshot' or 'config'.")

## Available snapshots

Each snapshot is a folder under `raw/scraping/snapshots/` containing
`manifest.json`, `html/`, `json/`, and per-ID request state in `state/`.
Status is `complete` (no failed IDs), `partial` (finished with failed IDs,
resumable), or `in_progress` (created or interrupted, resumable).

In [ ]:
list_snapshots()

## Optional collection

A `"new"` collection always creates a new snapshot folder; existing snapshots
are never modified. `"resume"` continues an unfinished snapshot with the same
ID range: stored records and unavailable IDs are skipped, failed IDs are
retried. Collection ends by rebuilding the snapshot's CSVs from its JSON.

In [ ]:
selected_snapshot = None
if collection_mode == "new":
    selected_snapshot = create_snapshot(label=snapshot_label)
elif collection_mode == "resume":
    selected_snapshot = resolve_snapshot(snapshot_id)

if selected_snapshot is not None:
    print("Collecting into snapshot", selected_snapshot.name)
    new_records = run_scraper(
        start_id=first_id,
        end_id=last_id,
        snapshot_dir=selected_snapshot,
        sleep_interval=request_delay_seconds,
        user_agent=os.getenv("HTTP_USER_AGENT") or "CartaSocialPipeline/1.0",
    )
    manifest = read_manifest(selected_snapshot)
    print("Status:", manifest["status"], manifest["counts"])
    print("Service records in the snapshot:", len(new_records))
else:
    print("No HTTP requests: using stored data.")

## Snapshot tables

Service records and the equipment ID/URL index are derived tables written to
`interim/scraping/<snapshot_id>/` with a `build.json` describing the rebuild.
Rebuilding never changes the snapshot. With `rebuild_from = "html"`, pages that
no longer match the parser are listed in `parse_error_ids`.

In [ ]:
snapshot_tables = None
if inspect_source == "snapshot" or rebuild_csv:
    if selected_snapshot is None:
        selected_snapshot = resolve_snapshot(snapshot_id)
    snapshot_tables = export_dir_for(selected_snapshot)
    if rebuild_csv or not (snapshot_tables / "equipments.csv").exists():
        export_records(selected_snapshot, source=rebuild_from)
    build = json.loads((snapshot_tables / "build.json").read_text(encoding="utf-8"))
    print("Snapshot:", selected_snapshot.name)
    print("Tables:", snapshot_tables)
    print({key: build[key] for key in ["source", "built_at", "service_rows", "equipment"]})
    if build["parse_error_ids"]:
        print("IDs whose HTML could not be parsed:", build["parse_error_ids"])
else:
    print("Inspecting configured inputs; no snapshot tables used.")

## Source inspection

The service CSV is read as UTF-8, with equipment IDs treated as text. Field
labels are standardized to English for inspection; source values and the
input file remain unchanged.

With `inspect_source = "snapshot"` the tables of the selected snapshot are
read directly; no configuration change or kernel restart is needed.
Notebook 02 still reads `inputs.services` from config.toml.

In [ ]:
if inspect_source == "snapshot":
    source_file = snapshot_tables / "equipments.csv"
    url_csv_file = snapshot_tables / "equipment_urls.csv"
    url_text_file = None
else:
    source_file = INPUTS["services"]
    url_csv_file = INPUTS["equipment_urls"]
    url_text_file = INPUTS["url_text"]

print("Service records file:", source_file)
source_records = pd.read_csv(source_file, dtype={"Equipment ID": "string"})
services = standardize_columns(source_records)
services.head()

In [ ]:
service_count = len(services)
equipment_count = services["equipment_id"].nunique()

print("Service records:", service_count)
print("Distinct equipment:", equipment_count)


## Initial quality checks

Missing values and exact duplicate rows are reported without automatic
removal. Repeated equipment IDs are expected when a facility offers more
than one service.

Source update dates and collection timestamps represent different events.
The input's provenance should establish which dates are available before
they are used to interpret change over time.


In [ ]:
missing_values = services.isna().sum()
exact_duplicate_count = services.duplicated().sum()

print(missing_values)
print("Exact duplicate rows:", exact_duplicate_count)


## Equipment URL index

The URL CSV contains equipment_id and url. The ID is validated against the
URL query parameter. The main service CSV remains unchanged. Several service
records can refer to one equipment URL.

In config mode, an optional text URL log can be cross-checked through
inputs.url_text. Snapshot URL indexes include equipment without service rows.
Coverage is computed below without contacting the source portal.

In [ ]:
from scripts.source_urls import load_equipment_urls

url_index = load_equipment_urls(csv_path=url_csv_file, txt_path=url_text_file)
service_equipment_ids = services["equipment_id"].drop_duplicates()
missing_url_ids = service_equipment_ids[~service_equipment_ids.isin(url_index["equipment_id"])]
unmatched_url_ids = url_index[~url_index["equipment_id"].isin(service_equipment_ids)]

print("Equipment URLs:", len(url_index))
print("Service equipment IDs without a URL:", len(missing_url_ids))
print("URL IDs without service rows:", len(unmatched_url_ids))
url_index.head()

## Processing handoff

Continue with [02_cleaning_geocoding.ipynb](02_cleaning_geocoding.ipynb)
to validate values, prepare equipment addresses, and match coordinate caches.
To process a snapshot there, point inputs.services and inputs.equipment_urls
to `interim/scraping/<snapshot_id>/equipments.csv` and `equipment_urls.csv`.
The observation unit and collection limitations are documented in
[the methodology](../docs/methodology.md).